# 01 · Data and validation

**Question:** What can these splits establish about unseen community rules?

Only two rule types have labels. We explicitly retain that limit throughout the research; thousands of candidate features cannot manufacture additional policy coverage.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Competition evidence | 2,029 rows | two labeled rules")
print("Aggregate checksums verified. This notebook performs no model fitting.")
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records, heldout=False):
    return pd.DataFrame([{"Representation": r["model"], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]}
        for r in records if not heldout or r["protocol"] == "heldout_rule"]).round(4)


Competition evidence | 2,029 rows | two labeled rules
Aggregate checksums verified. This notebook performs no model fitting.


In [2]:
import sys
sys.path.insert(0, str(root / "scripts"))
from build_research_report import display_figure
from jigsaw_rules.features import feature_evidence
from jigsaw_rules.research import research_evidence
from jigsaw_rules.diagnostics import diagnostic_evidence
from jigsaw_rules.pairs import pairs_evidence
from jigsaw_rules.robustness import robustness_evidence
from jigsaw_rules.gate import feature_gate
from jigsaw_rules.instructions import instruction_evidence

controls = feature_evidence(root)
research = research_evidence(root)
sensitivity = diagnostic_evidence(root)
pairs = pairs_evidence(root)
robustness = robustness_evidence(root)
assert all(item is not None for item in (controls, research, sensitivity, pairs, robustness))
gate = feature_gate(root)
instructions = instruction_evidence(root)
print("Verified research runs:", gate["studies"])


Verified research runs: {'research': '9ec008d506b0bc64a717', 'sensitivity': 'bce07fd60bc7543fca49', 'pairs': '537cf2213c813b8ebd4b', 'robustness': 'a09455ec14e9b3d6ab61', 'instructions': '96bf69f42f9063e01a12'}


## Rule and duplicate audit
Count and violation prevalence are reported together. Repeated comments are grouped before splitting. Preview-test overlap makes that file useful for schema checks only.

In [3]:
audit = baseline["audit"]
by_rule = pd.DataFrame(audit["by_rule"])
by_rule["rule"] = by_rule["rule"].str.split(":").str[0]
display(by_rule.round(4))
display(pd.DataFrame({"Finding": ["Duplicate training bodies", "Train / preview-test overlap", "Comment equals its own support example"], "Rows": [audit["duplicate_training_bodies"], audit["train_test_body_overlap"], sensitivity["audit"]["self_support_rows"]]}))

,rule,size,mean
0,No Advertising,1012,0.4328
1,No legal advice,1017,0.5831


,Finding,Rows
0,Duplicate training bodies,162
1,Train / preview-test overlap,10
2,Comment equals its own support example,18


## Two validation questions
**Familiar-rule CV** stratifies by rule and target, grouping normalized duplicate comments. **Held-out-rule CV** excludes the evaluated rule from training. Both purge any training row whose comment or supplied examples contain a validation comment. The research reuses the preserved row assignments.

This is stringent: the familiar-rule folds retain only 237–287 training rows. Sparse fold estimates are a real limitation, not grounds for weakening the boundary.

In [4]:
split_audit = pd.DataFrame(robustness["audit"]["folds"])
display(split_audit.rename(columns={"before": "Exact-purged train rows", "after": "Near-copy-purged train rows"}))

,protocol,fold,Exact-purged train rows,Near-copy-purged train rows,removed,validation_rows,affected_validation_rows
0,seen_rule,0,237,237,0,676,0
1,seen_rule,1,287,287,0,677,0
2,seen_rule,2,279,279,0,676,0
3,heldout_rule,0,961,961,0,1012,0
4,heldout_rule,1,975,975,0,1017,0


## Learned features obey the same boundary
Vocabulary, IDF, scalers, empirical ranks, feature screening and NB token weights use outer training rows only. Target/context encodings additionally use three inner comment-group folds, purging inner validation comments from training comments and examples; the prior is fitted inside each inner fold. Unseen groups fall back to training priors.

Provided positive/negative examples are legitimate per-row inputs. Their semantics are not the current row's unknown target. Frozen encoders use no competition-label fitting. The 18 self-support rows are excluded in a separate scoring sensitivity analysis.

## Approximate-copy stress test
A second pass requires character-ngram cosine ≥0.95, token-set Jaccard ≥0.90 and at least 40 characters, with a fixed hashing representation. It uses text only, removes training rows, and keeps validation rows fixed. Thresholds were not optimized on outcomes. No additional copies met both thresholds after exact purging; this does **not** prove paraphrase or shared-origin isolation.

In [5]:
print(robustness["audit"]["interpretation"])
display(metric_table(robustness["results"], heldout=True))

Additional training-row removal only; original validation rows retained. Fixed thresholds were not selected with outcomes. This is not paraphrase isolation.


,Representation,Validation,Rule macro AUC,Log loss,Brier,Average precision
0,reference_near_purged,Held-out rule,0.6156,0.6736,0.2405,0.6241
1,character_near_purged,Held-out rule,0.6235,0.6646,0.2361,0.6200
2,original_reference,Held-out rule,0.6156,0.6736,0.2405,0.6241


## Inference scope
Temporal, rolling, lag, season, team, opponent and coaching variables are unavailable or inapplicable. Row order is not time. There is no legitimate external ranking system for these comments. External model weights are pinned and license documented; competition-specific permission for any future data augmentation must be verified before use.

The project computes **rule macro ROC AUC** as its local approximation to the competition's column-averaged AUC description. Pooled AUC is separate. No official scoring implementation or independent Kaggle score has confirmed equivalence.

Continue to [02 · Feature research](02_baseline_and_review.ipynb).